# 8.4. Multi-Branch Networks \(GoogLeNet\)

GoogLeNet was the winner to the 2014 ImageNet challenge. It proposed 2 revolutionary ideas of its time which distinguished it from other candidates to the 2014 challenge which continue to influence the development of CNNs to this day.

1. Instead of stacking convolutions in a purely sequential manner, branch out and calculate convolutions with different kernel sizes in parallel and concatenate the results along the channel dimension
1. Design the network to have distinct _stem_ \(data ingestion\), _body_ \(data processing\) and _head_ \(prediction\) structures, a recurring theme in modern neural networks that persist to this day

Regarding \(1\), as the age-old saying goes:

> Only children make choices, adults want it all!

This notebook is based on the [AtomGit AI Notebook Lab](https://ai.gitcode.com/docs/notebooks/free-usage/) cloud environment with datacenter-grade Ascend 910B4 training-optimized NPUs. The software versions used in this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.11
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 0     910B4               | OK            | 100.9       40                0    / 0             |
| 0                         | 0000:C1:00.0  | 0           0    / 0          2860 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

In [2]:
%pip install mindspore==2.8.0 \
    -i https://repo.mindspore.cn/pypi/simple \
    --trusted-host repo.mindspore.cn \
    --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.8.0


[WARNING] DEVICE(15623,ffff1ee5f120,python3.11):2026-05-12-23:43:36.931.065 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:176] CheckVmmDriverVersion] Open file /etc/ascend_install.info failed.
[WARNING] DEVICE(15623,ffff1ee5f120,python3.11):2026-05-12-23:43:36.931.113 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:204] CheckVmmDriverVersion] Open file /usr/local/Ascend/driver/version.info failed.


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.4.1. Inception Blocks

At the heart of GoogLeNet is _inception blocks_. The name is derived from the movie [Inception](https://en.wikipedia.org/wiki/Inception) where "we need to go deeper". Each inception block computes the following convolutions in parallel before concatenating the results along the channel dimension.

1. A simple 1 by 1 convolution acting as a per-pixel FC layer along the channel dimension, introducing local nonlinearities
1. A 1 by 1 convolution with reduced channel dimensions to reduce the model's complexity, followed by a 3 by 3 convolution with an increased number of channels
1. A 1 by 1 convolution with reduced channel dimensions to reduce the model's complexity, followed by a 5 by 5 convolution with an increased number of channels
1. A max pooling layer with 3 by 3 pooling window, stride of 1 and padding of `1px`, followed by a 1 by 1 convolution

Below we implement our inception block as a custom layer. While not present in the original implementation, here we add batch normalization to each convolution after ReLU activation to stabilize the training of our neural network.

In [4]:
import mindspore.nn as nn
import mindspore.ops as ops

class Inception(nn.Cell):
    def __init__(self, in_c0, out_c1, out_c2, out_c3, out_c4, **kwargs):
        super(Inception, self).__init__(**kwargs)
        self.b1 = nn.SequentialCell([
            nn.Conv2d(in_c0, out_c1, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_c1)
        ])
        self.b2 = nn.SequentialCell([
            nn.Conv2d(in_c0, out_c2[0], kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_c2[0]),
            nn.Conv2d(out_c2[0], out_c2[1], kernel_size=3),
            nn.ReLU(),
            nn.BatchNorm2d(out_c2[1])
        ])
        self.b3 = nn.SequentialCell([
            nn.Conv2d(in_c0, out_c3[0], kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_c3[0]),
            nn.Conv2d(out_c3[0], out_c3[1], kernel_size=5),
            nn.ReLU(),
            nn.BatchNorm2d(out_c3[1])
        ])
        self.b4 = nn.SequentialCell([
            nn.MaxPool2d(kernel_size=3, pad_mode='same'),
            nn.Conv2d(in_c0, out_c4, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_c4)
        ])

    def construct(self, X):
        return ops.cat([
            self.b1(X),
            self.b2(X),
            self.b3(X),
            self.b4(X)
        ], axis=1)

Let's create an inception block `inception_1` and inspect the output shape from the entire block. We'll be using it shortly.

In [5]:
inception_1 = Inception(192, 64, (96, 128), (16, 32), 32)
inception_1

Inception(
  (b1): SequentialCell(
    (0): Conv2d(input_channels=192, output_channels=64, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff2d9ad210>, bias_init=None, format=NCHW)
    (1): ReLU()
    (2): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=b1.2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=b1.2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=b1.2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=b1.2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
  )
  (b2): SequentialCell(
    (0): Conv2d(input_channels=192, output_channels=96, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff

In [6]:
X = ops.randn(1, 192, 56, 56)
y_hat = inception_1(X)
print(f'Input shape: {X.shape}')
print(f'Output shape: {y_hat.shape}')

Input shape: (1, 192, 56, 56)
Output shape: (1, 256, 56, 56)


## 8.4.2. GoogLeNet Model

Quoting directly from [chapter 8.4 of D2L](https://d2l.ai/chapter_convolutional-modern/googlenet.html):

> GoogLeNet uses a stack of a total of 9 inception blocks, arranged into three groups with max-pooling in between, and global average pooling in its head to generate its estimates. Max-pooling between inception blocks reduces the dimensionality. At its stem, the first module is similar to AlexNet and LeNet.

Let's implement GoogLeNet piece by piece. Instead of defining our own subclass of `mindspore.nn.Cell` for GoogLeNet, we'll simply define each individual block from GoogLeNet and string them together with `mindspore.nn.SequentialCell`.

The stem consists of 2 modules.

1. The 1st module uses a 7 by 7 convolutional layer with 64 output channels
1. The 2nd module uses 2 convolutional layers.
    1. 1 by 1 convolutional layer with 64 output channels
    1. 3 by 3 convolutional layer tripling the number of output channels to 192

The body consists of 2 modules.

1. 2 inception blocks in sequence followed by max pooling
1. 5 inception blocks in sequence followed by max pooling

The head consists of 1 module with 2 inception blocks in sequence followed by global average pooling and a flattening layer similar to NiN. Finally, the output of the flattening layer is passed through an FC layer to map to 1024 input channels to 10 raw logits corresponding to our class labels.

In [7]:
googlenet = nn.SequentialCell([
    # 1st module
    nn.SequentialCell([
        nn.Conv2d(1, 64, kernel_size=7, stride=2),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(3, stride=2, pad_mode='same')
    ]),
    # 2nd module
    nn.SequentialCell([
        nn.Conv2d(64, 64, kernel_size=1),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.Conv2d(64, 192, kernel_size=3),
        nn.ReLU(),
        nn.BatchNorm2d(192),
        nn.MaxPool2d(3, stride=2, pad_mode='same')
    ]),
    # 3rd module
    nn.SequentialCell([
        inception_1, # Reuse our 1st inception block defined earlier
        Inception(256, 128, (128, 192), (32, 96), 64),
        nn.MaxPool2d(3, stride=2, pad_mode='same')
    ]),
    # 4th module
    nn.SequentialCell([
        Inception(480, 192, (96, 208), (16, 48), 64),
        Inception(512, 160, (112, 224), (24, 64), 64),
        Inception(512, 128, (128, 256), (24, 64), 64),
        Inception(512, 112, (144, 288), (32, 64), 64),
        Inception(528, 256, (160, 320), (32, 128), 128),
        nn.MaxPool2d(3, stride=2, pad_mode='same')
    ]),
    # 5th module
    nn.SequentialCell([
        Inception(832, 256, (160, 320), (32, 128), 128),
        Inception(832, 384, (192, 384), (48, 128), 128),
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten()
    ]),
    # Final linear layer mapping to our 10 class labels
    nn.Dense(1024, 10)
])
googlenet

SequentialCell(
  (0): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(7, 7), stride=(2, 2), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff1f04be10>, bias_init=None, format=NCHW)
    (1): ReLU()
    (2): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
    (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=SAME)
  )
  (1): SequentialCell(
    (0): Conv2d(input_channels=64, output_channels=64, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<

Let's pass in a grayscale image of 96 by 96 pixels and inspect the shape of the outputs from each successive module.

In [8]:
def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 96, 96)
layer_summary(net=googlenet, X_shape=X_shape)

Input shape: (1, 1, 96, 96)
Output shape from SequentialCell: (1, 64, 24, 24)
Output shape from SequentialCell: (1, 192, 12, 12)
Output shape from SequentialCell: (1, 480, 6, 6)
Output shape from SequentialCell: (1, 832, 3, 3)
Output shape from SequentialCell: (1, 1024)
Output shape from Dense: (1, 10)


## 8.4.3. Training

Let's train our implementation of GoogLeNet on the Fashion MNIST dataset upscaled to 96 by 96 pixels.

In [9]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [10]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [11]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [12]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(96, 96)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [13]:
import mindspore.amp as amp

googlenet_amp = amp.auto_mixed_precision(network=googlenet, amp_level='O2')
googlenet_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): SequentialCell(
      (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(7, 7), stride=(2, 2), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff1f04be10>, bias_init=None, format=NCHW)
      (1): ReLU()
      (2): _OutputTo16(
        (_backbone): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
      )
      (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=SAME)
    )
    (1): SequentialCell(
      (0): Conv2d(input_channels=64, output_channels=64, kernel_size=(1, 1), stride=(1,

In [14]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [15]:
optimizer = nn.SGD(params=googlenet_amp.trainable_params(), learning_rate=0.01)
optimizer

SGD()

In [16]:
loss_scale_manager = amp.FixedLossScaleManager(loss_scale=1024.0)
loss_scale_manager

In [17]:
from mindspore.train import Model

model = Model(network=googlenet_amp,
              loss_fn=loss_fn,
              optimizer=optimizer,
              metrics={'accuracy', 'loss'},
              loss_scale_manager=loss_scale_manager)
model

In [18]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [19]:
epochs = 10

In [20]:
model.fit(epoch=epochs,
          train_dataset=train_ds,
          valid_dataset=test_ds,
          callbacks=[early_stopping],
          dataset_sink_mode=True)

path string is NULLpath string is NULL......Restoring model weights from the end of the best epoch.
Epoch 00010: early stopping


Let's check the validation loss and accuracy of our trained NiN model against the Fashion MNIST dataset.

In [21]:
metrics = model.eval(valid_dataset=test_ds)
val_acc = metrics['accuracy']
val_loss = metrics['loss']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')

Validation loss: 0.2409
Validation accuracy: 0.9147


Our trained GoogLeNet model reaches $90\%$ accuracy easily - not bad!

## 8.4.4. Discussion

We saw in this chapter how GoogLeNet was designed and implemented and its main innovations that continue to influence modern neural network design to this day.

In the next chapter, let's take a closer look at batch normalization - what it is, how it works and when to use it.